In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = ''
df = pd.read_csv(file_path, nrows=5000)

In [ ]:
df.head()

,Nome da Música,Artista,Gênero Musical,Letra da Música
0,Carolina,Seu Jorge,MPB,Carolina é uma menina bem difícil de esquecer ...
1,Epitáfio,Titãs,Rock,Devia ter amado mais Ter chorado mais Ter vist...
2,Lugar Ao Sol,Charlie Brown Jr.,Rock,"Que bom viver, como é bom sonhar E o que ficou..."
3,Relicário - Ao Vivo,Cássia Eller,Rock,É uma índia com colar A tarde linda que não qu...
4,Você Me Vira A Cabeça (Me Tira Do Sério),Alcione,Samba,"Você me vira a cabeça, me tira do sério Destró..."


In [ ]:
import spacy

!python -m spacy download pt_core_news_lg
pln = spacy.load('pt_core_news_lg')
pln

from spacy.tokens import Doc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 1.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
Doc.set_extension("musica_id", default=None, force=True)

In [ ]:
sujeitos_femininos = ["mulher", "mulheres", "ela", "elas", "menina", "meninas", "garota", "garotas", "senhora", "senhoras", "senhorita", "senhoritas", "moça", "moças", "donzela", "donzelas", "dama", "damas", "rainha", "rainhas", "esposa", "esposas", "namorada", "namoradas", "mina", "minas", "mãe", "mães", "filha", "filhas", "tia", "tias", "avó", "avós", "neta", "netas", "sobrinha", "sobrinhas", "madrasta", "madrastas", "entendeada", "entedeadas", "musa", "musas", "diva", "divas", "deusa", "deusas", "querida", "queridas", "princesa", "princesas"]

Teste

In [ ]:
def define_pattern(subject_array, gender: Literal["Masc", "Fem"]):
  matcher = Matcher(vocab=pln.vocab)
  gender_morph = f"Gender={gender}|Number=Sing"

  #Sujeitos pré definidos
  sujeitoauxadjetivo = [
    {'LOWER': {'IN': sujeitos_femininos}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX',  'OP': '?'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['ADJ']}},  # Adjetivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  sujeitoauxsubst = [
    {'LOWER': {'IN': sujeitos_femininos}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'AUX'},  # Verbo auxiliar
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['NOUN']}, "MORPH": {"IN": [gender_morph]}},  # Substantivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  sujeitoauxverbo = [
    {'LOWER': {'IN': sujeitos_femininos}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX',  'OP': '?'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['VERB'], "MORPH": "Gender=Fem|Number=Sing|VerbForm=Part|Voice=Pass"}},  # Verbo na Voz Passiva
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  #Nome Próprio
  nomeauxproprio = [
      {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": [gender_morph]}},
      {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
      {'POS': 'ADV', 'OP': '?'},  # Negação
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX'},  # Verbo auxiliar
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
      {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
      {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
      {'POS': {'IN': ['ADJ']}},  # Adjetivo
      {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
      {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
      {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
      {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]


  nomeauxsubst = [
      {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": [gender_morph]}},
      {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
      {'POS': 'ADV', 'OP': '?'},  # Negação
      {'POS': 'AUX'},  # Verbo auxiliar
      {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
      {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
      {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
      {'POS': {'IN': ['NOUN']}, "MORPH": {"IN": [gender_morph]}},  # Substantivo
      {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
      {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
      {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
      {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  nomeauxverbo = [
      {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": [gender_morph]}},
      {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
      {'POS': 'ADV', 'OP': '?'},  # Negação
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX'},  # Verbo auxiliar
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
      {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
      {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
      {'POS': {'IN': ['VERB'], "MORPH": "Gender=Fem|Number=Sing|VerbForm=Part|Voice=Pass"}},  # Verbo na Voz Passiva
      {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
      {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
      {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
      {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  matcher.add('sujeitoauxadjetivo', patterns=[sujeitoauxadjetivo])
  matcher.add('nomeauxproprio', patterns=[nomeauxproprio])
  matcher.add('sujeitoauxsubst', patterns=[sujeitoauxsubst])
  matcher.add('nomeauxsubst', patterns=[nomeauxsubst])
  matcher.add('sujeitoauxverbo', patterns=[sujeitoauxverbo])
  matcher.add('nomeauxverbo', patterns=[nomeauxverbo])

  return matcher


In [ ]:
from spacy.matcher import

matcher = Matcher(vocab=pln.vocab)

#Sujeitos pré definidos
sujeitoauxadjetivo = [
    {'LOWER': {'IN': sujeitos_femininos}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX',  'OP': '?'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['ADJ']}},  # Adjetivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
]

sujeitoauxsubst = [
    {'LOWER': {'IN': sujeitos_femininos}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'AUX'},  # Verbo auxiliar
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['NOUN']}, "MORPH": {"IN": ['Gender=Fem|Number=Sing']}},  # Adjetivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
]

sujeitoauxverbo = [
    {'LOWER': {'IN': sujeitos_femininos}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX',  'OP': '?'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['VERB'], "MORPH": "Gender=Fem|Number=Sing|VerbForm=Part|Voice=Pass"}},  # Verbo na Voz Passiva
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
]



#Nome Próprio
nomeauxproprio = [
    {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": ['Gender=Fem|Number=Sing']}},
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['ADJ']}},  # Adjetivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'PROPN']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
]

nomeauxsubst = [
    {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": ['Gender=Fem|Number=Sing']}},
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'AUX'},  # Verbo auxiliar
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['NOUN']}, "MORPH": {"IN": ['Gender=Fem|Number=Sing']}},  # Substantivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
]


matcher.add('sujeitoauxadjetivo', patterns=[sujeitoauxadjetivo])
matcher.add('nomeauxproprio', patterns=[nomeauxproprio])
matcher.add('sujeitoauxsubst', patterns=[sujeitoauxsubst])
matcher.add('nomeauxsubst', patterns=[nomeauxsubst])

In [ ]:
import re

def preprocessar_texto(texto):
    texto = re.sub(r"\n(?!\n)", ". ", texto) #Substitui quebra de linha por ponto
    return texto

In [ ]:
resultados = []

for index, row in df.iterrows():
    texto_original = str(row['Letra da Música'])  # Texto original
    texto_processado = preprocessar_texto(texto_original)  # Pré-processamento

    # segmentação correta de frases
    doc = pln(texto_processado)

    # id único ao doc
    doc._.musica_id = row['Nome da Música']

    matches = matcher(doc)

    # para pegar a maior versão da frase
    matches_filtrados = {}

    for match_id, start_idx, end_idx in matches:
        palavras = doc[start_idx:end_idx].text
        key = (doc._.musica_id, palavras.lower())

        if doc[start_idx].sent != doc[end_idx - 1].sent:
            continue

        # Se já temos essa expressão na mesma música armazenamos apenas a versão maior
        if key not in matches_filtrados or len(palavras) > len(matches_filtrados[key]):
            matches_filtrados[key] = palavras

    expressoes_unicas = sorted(set(matches_filtrados.values()), key=lambda x: -len(x))

    expressoes_finais = []
    for exp in expressoes_unicas:
        if not any(exp in maior for maior in expressoes_finais):
            expressoes_finais.append(exp)

    expressoes_finais = sorted(expressoes_finais, key=lambda x: doc.text.find(x))

    for expressao in expressoes_finais:
      print(f"Música: {doc._.musica_id} | Expressão encontrada: {expressao}")
      resultados.append({
          "Frase": expressao,
          "Nome da Música": row['Nome da Música'],
          "Artista": row['Artista'],
          "Letra da Música": row['Letra da Música'],
          "Gênero Musical": row['Gênero Musical']
      })

resultados_df = pd.DataFrame(resultados)

Música: Carolina | Expressão encontrada: menina bem difícil de esquecer
Música: Carolina | Expressão encontrada: ela é muito sensual
Música: Carolina | Expressão encontrada: Menina bela
Música: Rodo cotidiano | Expressão encontrada: Ela é linda
Música: Ela Vai Voltar (Todos Os Defeitos De Uma Mulher Perfeita) | Expressão encontrada: Ela é guerreira
Música: Ela Vai Voltar (Todos Os Defeitos De Uma Mulher Perfeita) | Expressão encontrada: ela é mulher de verdade
Música: Ela Vai Voltar (Todos Os Defeitos De Uma Mulher Perfeita) | Expressão encontrada: Ela é discreta
Música: Ai Se Eu Te Pego | Expressão encontrada: menina mais linda
Música: Meiga e abusada | Expressão encontrada: menina brincalhona
Música: Envolvidão | Expressão encontrada: Ela é inacreditável
Música: Envolvidão | Expressão encontrada: ela é fácil rapaz
Música: Envolvidão | Expressão encontrada: mulheres vulgares
Música: Ela É do Tipo | Expressão encontrada: menina mete muito gostoso
Música: Amor de Fim de Noite | Expressã

In [ ]:
resultados_df.to_csv('resultados.csv', index=False)
from google.colab import files
files.download('resultados.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
resultados.shape